In [2]:
import openai
import instructor
from pydantic import BaseModel,Field
from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

RAG Pipeline

In [3]:
client=instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [4]:
class RAG_GenerationResponse(BaseModel):
    answer:str=Field(description="Answer to the question")

In [6]:
qdrant_client = QdrantClient(url="http://localhost:6333")


def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
   
    return response.data[0].embedding   


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt




def generate_answer(prompt):

    response,raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort":"none"},
        response_model=RAG_GenerationResponse
    )
    
    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer= {

        "answer":answer.answer,
        "question":question,
        "retrieved_context_ids":retrieved_context["retrieved_context_ids"],
        "retrieved_context":retrieved_context["retrieved_context"]
    }
    return final_answer

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [7]:
output=rag_pipeline("any earbud")

In [8]:
output

{'answer': 'Yes—there are several earbuds available:\n1) Jesebang Wireless Earbud (Bluetooth 5.3, up to 40H, ENC mic, USB-C LED display, IP7 waterproof, sport earhooks) – Black (Rating 4.9)\n2) LUDOS Turbo Wired Earbuds (in-ear ergonomic, memory foam, microphone, durable cable, bass) (Rating 4.4)\n3) Earbuds Cleaning Pen / 3-in-1 Earpods Cleaning Kit (pen tip + soft brush + flocking sponge for charging case and sound outlet holes) (Rating 4.3)\n4) 2-Pack Wired Earbuds with Lightning Connector for iPhone (built-in mic + volume control) (Rating 4.1)',
 'question': 'any earbud',
 'retrieved_context_ids': ['B0C2TLBSMP',
  'B09XB59YPR',
  'B07VSWK14G',
  'B0C6KM4H9M',
  'B0CCK3KL19'],
 'retrieved_context': ['Jesebang Wireless Earbud, Bluetooth Headphones 5.3 Stereo Earphones 2023 Ear Buds 40H ENC Mic, in-Ear Earbud USB-C LED Display IP7 Waterproof Sport Earhooks Headset, Black ',
  'Earbuds Cleaning Pen,Multifunctional Airpods Cleaner,3-in-1 Earpods Cleaning Kit with Soft Brush Apply to Air

RAG Pipeline with Grounding Context

In [9]:
class RAGUsedContext(BaseModel):
    id:str=Field(description="ID of item used to Answer the question")
    description:str=Field(description="Description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer:str=Field(description="Answer to the question")
    references:list[RAGUsedContext]=Field(description= "List of items used to answer this question")

In [10]:


def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
   
    return response.data[0].embedding   


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- if you are describing multiple products, list them out as a list.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt




def generate_answer(prompt):

    response,raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort":"none"},
        response_model=RAGGenerationResponse
    )
    
    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer= {
        "data_object":answer,
        "answer":answer.answer,
        "references":answer.references,
        "question":question,
        "retrieved_context_ids":retrieved_context["retrieved_context_ids"],
        "retrieved_context":retrieved_context["retrieved_context"]
    }
    return final_answer

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [11]:
output=rag_pipeline("Do you have earphones")

In [12]:
output

{'data_object': RAGGenerationResponse(answer='Yes—here are earphones/earbuds available:\n\n- Jesebang Wireless Earbud, Bluetooth Headphones 5.3 Stereo Earphones (Black) — 40H ENC mic, in-ear, USB‑C LED display, IP7 waterproof, sport earhooks\n- LUDOS Turbo Wired Earbuds with Microphone — ergonomic in-ear, memory foam, durable cable\n- 2 Pack Apple Headphones Wired Earbuds with Lightning Connector — built-in microphone & volume control\n\nIf you tell me whether you want wireless or wired (and phone type), I can narrow it down.', references=[RAGUsedContext(id='B0C2TLBSMP', description='Jesebang Wireless Earbud, Bluetooth Headphones 5.3 Stereo Earphones 2023 Ear Buds 40H ENC Mic, in-Ear Earbud USB-C LED Display IP7 Waterproof Sport Earhooks Headset, Black'), RAGUsedContext(id='B07VSWK14G', description='LUDOS Turbo Wired Earbuds in-Ear Ergonomic Earphones with Microphone, Memory Foam, Durable Cable, Bass'), RAGUsedContext(id='B0CCK3KL19', description='2 Pack-Apple Headphones Wired Earbuds 

In [13]:
print(output["answer"])

Yes—here are earphones/earbuds available:

- Jesebang Wireless Earbud, Bluetooth Headphones 5.3 Stereo Earphones (Black) — 40H ENC mic, in-ear, USB‑C LED display, IP7 waterproof, sport earhooks
- LUDOS Turbo Wired Earbuds with Microphone — ergonomic in-ear, memory foam, durable cable
- 2 Pack Apple Headphones Wired Earbuds with Lightning Connector — built-in microphone & volume control

If you tell me whether you want wireless or wired (and phone type), I can narrow it down.


In [15]:
output=rag_pipeline("Do you have earphones",top_k=10)

In [16]:
output

{'data_object': RAGGenerationResponse(answer='Yes—here are the earphones/earbuds we have available:\n\n- Jesebang Wireless Earbud (Bluetooth 5.3), stereo earphones with 40H battery, ENC mic, USB-C LED display, IP7 waterproof, sport earhooks (Black)\n- LUDOS Turbo Wired Earbuds, in-ear ergonomic with microphone and memory foam\n- 2 Pack Wired Earbuds with Lightning Connector (built-in microphone & volume control) compatible with iPhone models\n', references=[RAGUsedContext(id='B0C2TLBSMP', description='Jesebang Wireless Earbud (Bluetooth 5.3) stereo earphones with ENC mic, 40H, USB-C LED display, IP7 waterproof, sport earhooks, Black'), RAGUsedContext(id='B07VSWK14G', description='LUDOS Turbo Wired Earbuds in-ear ergonomic with microphone and memory foam'), RAGUsedContext(id='B0CCK3KL19', description='2 Pack Apple Headphones Wired Earbuds with Lightning Connector, built-in microphone & volume control')]),
 'answer': 'Yes—here are the earphones/earbuds we have available:\n\n- Jesebang Wi

In [17]:
print(output["answer"])

Yes—here are the earphones/earbuds we have available:

- Jesebang Wireless Earbud (Bluetooth 5.3), stereo earphones with 40H battery, ENC mic, USB-C LED display, IP7 waterproof, sport earhooks (Black)
- LUDOS Turbo Wired Earbuds, in-ear ergonomic with microphone and memory foam
- 2 Pack Wired Earbuds with Lightning Connector (built-in microphone & volume control) compatible with iPhone models



In [18]:
output=rag_pipeline("Do you have earphones",top_k=15)

In [28]:
output

{'data_object': RAGGenerationResponse(answer='Yes—here are earphones/earbuds that are available:\n- Jesebang Wireless Earbud (Bluetooth 5.3, 40H ENC mic, USB‑C LED display, IP7 waterproof, black)\n- LUDOS Turbo Wired Earbuds (in-ear ergonomic, memory foam, with microphone)\n- 2 Pack Wired Earbuds with Lightning connector (built-in microphone & volume control, compatible with iPhone Lightning devices)\n', references=[RAGUsedContext(id='B0C2TLBSMP', description='Jesebang Wireless Earbud, Bluetooth Headphones 5.3 Stereo Earphones 2023 Ear Buds 40H ENC Mic, in-Ear Earbud USB-C LED Display IP7 Waterproof Sport Earhooks Headset, Black'), RAGUsedContext(id='B07VSWK14G', description='LUDOS Turbo Wired Earbuds in-Ear Ergonomic Earphones with Microphone, Memory Foam, Durable Cable, Bass'), RAGUsedContext(id='B0CCK3KL19', description='2 Pack-Apple Headphones Wired Earbuds with Lightning Connector Earphones Built-in Microphone & Volume Control')]),
 'answer': 'Yes—here are earphones/earbuds that a

In [19]:
print(output["answer"])

Yes—here are earphones/earbuds currently available:
- Jesebang Wireless Earbud (Bluetooth 5.3, 40H, ENC mic, USB-C LED display, IP7 waterproof, black)
- LUDOS Turbo Wired Earbuds with microphone (memory foam, durable cable)
- 2 Pack Wired Earbuds with Lightning connector for iPhone (built-in mic & volume control)
- Earbuds Cleaning Pen / 3-in-1 cleaning kit (accessory for earbuds)

Which type do you want: wireless Bluetooth earbuds or wired earbuds (3.5mm or Lightning)?
